# Mini project: Forecasting Apple's quarterly revenue

**Question:** can a statistical model forecast quarterly revenue better than a simple baseline, on data it has not seen?

**Flow**
1. Load and clean the data
2. Look at the data (trend and seasonality)
3. Split into train and test (last 8 quarters = test)
4. Baselines: naive, seasonal naive, moving average
5. Models: regression (trend + quarter), Holt-Winters, SARIMA
6. Compare errors (MAE, RMSE, MAPE) on the test set
7. Rolling cross-validation (check the result is not a one-off)
8. Forecast the next 4 quarters with the best model

**Data:** download the *quarterly* "Revenue from Business Activities - Total" for AAPL (same way as the annual file), save as `AAPL_Quarterly_Revenue.xlsx`. At least 10 years (40 quarters).

## Step 1: Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import statsmodels.formula.api as smf
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

## Step 2: Load the data

In [ ]:
df = pd.read_excel("AAPL_Quarterly_Revenue.xlsx", index_col=0)
df.head()

## Step 3: Clean the data
Rename the revenue column, convert dates, sort from oldest to newest, and create a quarterly index.

In [ ]:
df = df.rename(columns={"Revenue from Business Activities - Total": "Revenue"})
df = df.reset_index(drop=True)

df["Date"] = pd.to_datetime(df.Date, format='%Y-%m-%dT%H:%M:%SZ')
df = df.sort_values("Date")

# One row per quarter: use the quarter as the index (e.g. 2024Q4)
df["Quarter_label"] = df["Date"].dt.to_period("Q")
df = df.set_index("Quarter_label")

df = df[["Date", "Revenue"]]
df["Revenue"] = df["Revenue"].astype(float)
df.tail()

## Step 4: Check the data
The annual file had a repeated value, so we check before modelling.

In [ ]:
# 1) Same revenue value appearing more than once?
print("Repeated revenue values:")
print(df[df["Revenue"].duplicated(keep=False)])

# 2) Missing quarters?
full_range = pd.period_range(df.index.min(), df.index.max(), freq="Q")
print("Missing quarters:", list(full_range.difference(df.index)))

# 3) Very large jump from one quarter to the next (more than 50%)?
change = df["Revenue"].pct_change()
print("Big quarter-on-quarter changes:")
print(df[change.abs() > 0.5])

print("Number of quarters:", len(df))

## Step 5: Look at the data

In [ ]:
ax = df["Revenue"].plot(figsize=(10, 4), marker="o")
ax.set_title("Apple quarterly revenue")
ax.set_ylabel("Revenue")

Apple's revenue trends up, and the same quarter is high every year (seasonality). Next we add a time trend `t` and quarter dummies so a regression can learn both.

In [ ]:
df["t"] = np.arange(len(df))                    # 0, 1, 2, ... time trend
q = df.index.quarter                            # 1, 2, 3 or 4
df["Q2"] = (q == 2).astype(int)                 # dummy variables (Q1 is the reference quarter)
df["Q3"] = (q == 3).astype(int)
df["Q4"] = (q == 4).astype(int)

# Average revenue by quarter (shows the seasonal pattern)
df.groupby(q)["Revenue"].mean()

## Step 6: Train / test split
The last 8 quarters are the test set. The models never see them when they are fitted.

In [ ]:
n_test = 8

train = df.iloc[:-n_test].copy()
test = df.iloc[-n_test:].copy()

print("Train:", train.index[0], "to", train.index[-1], "(", len(train), "quarters )")
print("Test: ", test.index[0], "to", test.index[-1], "(", len(test), "quarters )")

## Step 7: Baseline forecasts
If a model cannot beat these, it is not worth using.

In [ ]:
last4 = train["Revenue"].iloc[-4:].values       # last 4 quarters of the training data

# Naive: next value = last value
test["Naive"] = train["Revenue"].iloc[-1]

# Seasonal naive: next value = the same quarter last year
test["SeasonalNaive"] = np.tile(last4, 10)[:n_test]

# Moving average of the last 4 quarters
test["MovingAvg4"] = last4.mean()

test[["Revenue", "Naive", "SeasonalNaive", "MovingAvg4"]]

## Step 8: Models

**Model A: Regression with trend and quarter dummies.** We use log(Revenue) because revenue grows by percentages, not by fixed amounts.

In [ ]:
reg = smf.ols(formula="np.log(Revenue) ~ t + Q2 + Q3 + Q4", data=train).fit()
reg.summary()

In [ ]:
test["Regression"] = np.exp(reg.predict(test))
test[["Revenue", "Regression"]]

**Model B: Holt-Winters** (exponential smoothing with trend and multiplicative seasonality).

In [ ]:
hw = ExponentialSmoothing(train["Revenue"].values,
                          trend="add",
                          seasonal="mul",
                          seasonal_periods=4,
                          initialization_method="estimated").fit()

test["HoltWinters"] = hw.forecast(n_test)
test[["Revenue", "HoltWinters"]]

**Model C: SARIMA.** A standard seasonal time-series model. We use one common setting, SARIMA(1,1,1)(0,1,1,4), fitted on the training data only.

In [ ]:
sarima = SARIMAX(train["Revenue"].values,
                 order=(1, 1, 1),
                 seasonal_order=(0, 1, 1, 4),
                 enforce_stationarity=False,
                 enforce_invertibility=False).fit(disp=False)

test["SARIMA"] = sarima.forecast(n_test)
test[["Revenue", "SARIMA"]]

## Step 9: Plot the test forecasts

In [ ]:
ax = train["Revenue"].iloc[-16:].plot(figsize=(10, 5), color="black", label="Train (last 16 quarters)")
test["Revenue"].plot(ax=ax, color="black", linewidth=3, label="Actual")
test["SeasonalNaive"].plot(ax=ax, label="Seasonal naive")
test["Regression"].plot(ax=ax, label="Regression")
test["HoltWinters"].plot(ax=ax, label="Holt-Winters")
test["SARIMA"].plot(ax=ax, label="SARIMA")
ax.legend()
ax.set_title("Forecasts vs actual revenue on the test set")

## Step 10: Compare the errors
- **MAE**: average error in revenue units
- **RMSE**: like MAE but punishes big misses more
- **MAPE**: average error as a % of actual revenue (easiest to explain)

In [ ]:
def mae(actual, pred):
    return np.mean(np.abs(actual - pred))

def rmse(actual, pred):
    return np.sqrt(np.mean((actual - pred) ** 2))

def mape(actual, pred):
    return np.mean(np.abs((actual - pred) / actual)) * 100

models = ["Naive", "SeasonalNaive", "MovingAvg4", "Regression", "HoltWinters", "SARIMA"]

results = pd.DataFrame(index=models, columns=["MAE", "RMSE", "MAPE %"], dtype=float)
for m in models:
    results.loc[m, "MAE"] = mae(test["Revenue"], test[m])
    results.loc[m, "RMSE"] = rmse(test["Revenue"], test[m])
    results.loc[m, "MAPE %"] = mape(test["Revenue"], test[m])

results = results.sort_values("MAPE %")
results.round(2)

## Step 11: Rolling cross-validation
One test period can be lucky or unlucky. So we repeat the exercise from 6 different starting points: train on the past, forecast the next 4 quarters, move forward, repeat. All models are packed in one function so we can reuse them.

In [ ]:
def get_forecasts(train_data, h):
    """Fit every model on train_data and return h-quarter-ahead forecasts."""
    out = {}
    last4 = train_data["Revenue"].iloc[-4:].values

    # Baselines
    out["Naive"] = np.repeat(train_data["Revenue"].iloc[-1], h)
    out["SeasonalNaive"] = np.tile(last4, h // 4 + 1)[:h]
    out["MovingAvg4"] = np.repeat(last4.mean(), h)

    # Regression: needs t and quarter dummies for the future quarters
    future = pd.DataFrame(index=pd.period_range(train_data.index[-1] + 1, periods=h, freq="Q"))
    future["t"] = np.arange(train_data["t"].iloc[-1] + 1, train_data["t"].iloc[-1] + 1 + h)
    future["Q2"] = (future.index.quarter == 2).astype(int)
    future["Q3"] = (future.index.quarter == 3).astype(int)
    future["Q4"] = (future.index.quarter == 4).astype(int)
    r = smf.ols(formula="np.log(Revenue) ~ t + Q2 + Q3 + Q4", data=train_data).fit()
    out["Regression"] = np.exp(r.predict(future)).values

    # Holt-Winters
    h_model = ExponentialSmoothing(train_data["Revenue"].values, trend="add", seasonal="mul",
                                   seasonal_periods=4, initialization_method="estimated").fit()
    out["HoltWinters"] = h_model.forecast(h)

    # SARIMA
    s_model = SARIMAX(train_data["Revenue"].values, order=(1, 1, 1), seasonal_order=(0, 1, 1, 4),
                      enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    out["SARIMA"] = s_model.forecast(h)

    return out

In [ ]:
horizon = 4
n_origins = 6
rows = []

for k in range(n_origins, 0, -1):
    end = len(df) - horizon * k           # the training data = first 'end' quarters
    if end < 20:                          # need enough history to fit the models
        continue
    train_cv = df.iloc[:end]
    test_cv = df.iloc[end:end + horizon]

    forecasts = get_forecasts(train_cv, horizon)
    for name, pred in forecasts.items():
        rows.append({"Origin": str(train_cv.index[-1]),
                     "Model": name,
                     "MAPE %": mape(test_cv["Revenue"].values, pred)})

cv = pd.DataFrame(rows)
cv_summary = cv.groupby("Model")["MAPE %"].mean().sort_values()
cv_summary.round(2)

In [ ]:
plt.figure(figsize=(8, 4))
plt.barh(cv_summary.index[::-1], cv_summary.values[::-1])
plt.xlabel("Average MAPE % in rolling cross-validation (lower is better)")
plt.title("Which model forecasts best?")
plt.show()

In [ ]:
baseline_mape = cv_summary["SeasonalNaive"]
best_model = cv_summary.index[0]
print("Best model:", best_model)
print("Its MAPE: %.1f%%  vs  seasonal naive baseline: %.1f%%" % (cv_summary[best_model], baseline_mape))
print("Change vs baseline: %.0f%%" % ((cv_summary[best_model] - baseline_mape) / baseline_mape * 100))

## Step 12: Forecast the next 4 quarters
Refit the best model on ALL the data and forecast ahead.

In [ ]:
final_fc = get_forecasts(df, 4)[best_model]

future_quarters = pd.period_range(df.index[-1] + 1, periods=4, freq="Q")
forecast_table = pd.DataFrame({"Forecast": final_fc.round(0)}, index=future_quarters)
forecast_table

In [ ]:
ax = df["Revenue"].iloc[-20:].plot(figsize=(10, 4), marker="o", label="History")
forecast_table["Forecast"].plot(ax=ax, marker="o", color="red", label="Forecast (" + best_model + ")")
ax.legend()
ax.set_title("Next 4 quarters")

## Step 13: Interpretation (fill in with YOUR numbers)

- Which model had the lowest cross-validated MAPE, and how does it compare with the seasonal naive baseline?
- Did the more complex models (Holt-Winters, SARIMA) beat the simple regression? Why might that be?
- What does the seasonal pattern look like (which quarter is highest)?
- What are the limits? (only about 10 years of data, shocks such as new products or COVID, the forecast is a single number with no uncertainty range)
- How could this help a business? (planning supply and inventory, comparing the forecast with actual sales each month)